## Chains
Chains are tehcniques to configure pipelines for an application by sequensing the individual task in series/parallel or conditionally.

### Type
* Series (check out output_parser.ipynb)
* Parallel
* Conditional


## Parallel Chain

<B>Example Statement:</B> Compare infrastructure between given two cities and generate a comparision report.


In [157]:
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda, RunnablePassthrough
from typing import Optional, Annotated, Any, List, Literal
from operator import itemgetter
from pydantic import Field, AnyUrl
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
# Model prepreation

model = init_chat_model(model='mistral-small-2603', 
                        model_provider='mistralai', 
                        temperature = 0.5 )

In [17]:
infraTemplate = PromptTemplate( template= ''' You are an govenment agent who has access to govenment infrastructure data.
                               Generate a infrastrure report for city {city} focusing on keypoints like {keypoints}.''',
                               input_variables=['city', 'keypoints'])

In [18]:
comparisionTemplate = PromptTemplate(template= 'Generate a comparision summary using report {report1} and {report2} for the cities.',
                                     input_variables=['report1', 'report2'])

In [6]:
strParser = StrOutputParser()

In [45]:
reportGenreration = RunnableParallel({  
    'report1' : {  "city": itemgetter("city1") , "keypoints"  : itemgetter("keypoints")} | infraTemplate | model | strParser ,
    'report2' : {  "city": itemgetter("city2") , "keypoints"  : itemgetter("keypoints")} | infraTemplate | model | strParser
})  

In [46]:
summarization = reportGenreration | comparisionTemplate | model | strParser

In [47]:
summarization

{
  report1: {
             city: RunnableLambda(itemgetter('city1')),
             keypoints: RunnableLambda(itemgetter('keypoints'))
           }
           | PromptTemplate(input_variables=['city', 'keypoints'], input_types={}, partial_variables={}, template=' You are an govenment agent who has access to govenment infrastructure data.\n                               Generate a infrastrure report for city {city} focusing on keypoints like {keypoints}.')
           | ChatMistralAI(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14', 'langchain-mistralai': '1.1.6'}}, output_version=None, profile={'name': 'Mistral Small 4', 'release_date': '2026-03-16', 'last_updated': '2026-03-16', 'open_weights': True, 'max_input_tokens': 256000, 'max_output_tokens': 256000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True

In [48]:
result = summarization.invoke( {'city1': 'Agra', 'city2': 'Chennai','keypoints' : ['Tourism', 'Road Infra', 'Education']} )

In [70]:
refined = result.split('\n')
for line in refined:
    line = line.replace(' |', '\t\t')
    print(line)

# **Comparative Summary: Government Infrastructure Reports – Agra (2024) vs. Chennai (2024)**

## **1. Overview**
Both **Agra** (a **heritage tourism hub**) and **Chennai** (a **commercial and educational center**) have undergone significant infrastructure developments in 2024. However, their priorities differ due to distinct economic drivers. Below is a structured comparison across **tourism, road infrastructure, and education**, highlighting **key achievements, challenges, and recommendations**.

---

## **2. Tourism Infrastructure**

| **Parameter**         		 **Agra (2024)**		 **Chennai (2024)**		 **Key Differences**		
|------------------------|----------------|-------------------|---------------------|
| **Primary Attractions**		 Taj Mahal (7.5M visitors), Agra Fort, Fatehpur Sikri		 Marina Beach, Fort St. George, Birla Planetarium, IT & corporate tourism		 Agra relies on **heritage tourism**, while Chennai balances **cultural, business, and coastal tourism**.		
| **Airport Upgrad

## Conditional Chains
<b>Example Statement: </b> Review and understand customer feedbacks, based on the sentimental analysis write a satisfactory response to negative feedbacks.

* Custoemr feedback -> Model -> [Positive , Negative, Nuetral] -> 
*                        ->  (if negative) [ Model -> Response -> Parser]
*                        ->  (else) -> End

In [114]:
from typing import TypedDict
class Emotion(TypedDict):
    review : str = Field("Customer review text ")
    emotion : Literal["Positive", "Negative", "Nuetral"] = Field(description='Emotion of the customer feedback')

In [115]:
model = init_chat_model(model="mistral-small-2603")
structuredModel = model.with_structured_output(Emotion)

In [116]:
feedbackTemplate = PromptTemplate( template= '''Anayse the emotions for the following customer review, respond for Positive, Negative, Neutral accordingly.' \
                                            Review : {review}''',
                                            input_variables=['review']
                                            ) 

In [154]:
def Positive(*args,**kwargs):
    print(*args, **kwargs)
    return '''Thanks, for the review, we really appriciate it.
            Best Regards,
            Support ABCD'''

In [150]:
responseTemplate = PromptTemplate( template = 'You are an customer chatbot, Write a satisfactory reply to the customer\'s review. Customer Reivew : {review},')

In [151]:
conditional = RunnableBranch(
    (lambda x : x['emotion'] == 'Negative', responseTemplate | model | strParser ), 
    (lambda x : x['emotion'] == 'Positive', RunnableLambda(Positive) ),
    RunnableLambda(lambda x :  x) )


In [152]:
chain = feedbackTemplate | structuredModel | conditional
chain

PromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, template="Anayse the emotions for the following customer review, respond for Positive, Negative, Neutral accordingly.'                                             Review : {review}")
| _ChatModelBinding(bound=ChatMistralAI(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14', 'langchain-mistralai': '1.1.6'}}, output_version=None, profile={'name': 'Mistral Small 4', 'release_date': '2026-03-16', 'last_updated': '2026-03-16', 'open_weights': True, 'max_input_tokens': 256000, 'max_output_tokens': 256000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': True, 'temperature': True}, client=<httpx.Client object at 0x75a0afd16570>, async_client=<httpx.AsyncClient object at 0x75a0afd16600>, mistral_ap

In [133]:
review1 = 'Love the product, must buy, value for money.'
review2 = 'Performance is fine, but very disappointed with battery as Drains withing 4 hours and heats up like lava. Need additional cooling fan while gaming,'
review3 = 'Camera quality is great, zooming stability could have been improved. Loved the sceen colors soothing and clear video playback. BUt some how batterty disappoints a bit, else good product.'

In [164]:
result  = chain.invoke({'review': review1})
result

{'review': 'Love the product, must buy, value for money.', 'emotion': 'Positive'}


'Thanks, for the review, we really appriciate it.Best Regards,Support ABCD'

In [165]:
result  = chain.invoke({'review': review2})
result

"Dear [Customer's Name],\n\nThank you for taking the time to share your feedback. We're glad to hear that you're satisfied with the overall performance of your device. However, we sincerely apologize for the inconvenience caused by the battery drainage and overheating issues you've experienced.\n\nWe understand how frustrating it can be to have a device that doesn't perform as expected, especially during gaming sessions. Your concerns have been shared with our technical team, and they will work towards finding a solution to improve your experience.\n\nIn the meantime, we would like to offer our assistance. Please don't hesitate to reach out to our customer support team at [customer support email/phone number] so they can guide you through any troubleshooting steps or discuss potential solutions.\n\nWe appreciate your patience and understanding, and we hope to have the opportunity to serve you better in the future.\n\nBest regards,\n\n[Your Name]\n[Your Position]\n[Company Name]"

In [155]:
result  = chain.invoke({'review': review3})
result

{'review': 'Camera quality is great, zooming stability could have been improved. Loved the sceen colors soothing and clear video playback. BUt some how batterty disappoints a bit, else good product.', 'emotion': 'Positive'}


'Thanks, for the review, we really appriciate it.Best Regards,Support ABCD'